<div align="center">

# Data Projects and Hackathon 3  
## Project 
Sergio Fernandez, Alessandro Mecchia 

</div>

In [ ]:
import os
import sys
import json
import time
import subprocess
import requests
from pathlib import Path

# Data Processing & Math
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.json as paj

# Machine Learning & NLP
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from langdetect import detect, DetectorFactory

# Visualization
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# Interactive Notebook Tools
import ipywidgets as widgets
from IPython.display import display

# Ensure consistent language detection
DetectorFactory.seed = 0
import duckdb 

In [ ]:
if torch.cuda.is_available():
    print(f"GPU found: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("GPU not detected. PyTorch is running on CPU.")

In [ ]:
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 5)

def is_empty(x):
    if x is None:
        return True
    if isinstance(x, str):
        return x.strip() == ""
    if isinstance(x, (list, tuple, set, dict)):
        return len(x) == 0
    if isinstance(x, np.ndarray):
        return x.size == 0
    return pd.isna(x)

## Functions 

In [ ]:
def pct(x): 
    return f"{x:,} ({x/n*100:.1f}%)"

def is_empty(val):
    if val is None:
        return True
    if isinstance(val, str):
        return val == ""
    if isinstance(val, (list, np.ndarray)):
        return len(val) == 0
    return False

In [ ]:
def infer_doc_type_local(row):
    title = row.get("title")
    venue = row.get("venue")

    title = title.lower().strip() if isinstance(title, str) else ""
    venue = venue.lower().strip() if isinstance(venue, str) else ""

    if any(x in title for x in ["thesis", "dissertation"]):
        return "dissertation"

    if "report" in title or "technical report" in title:
        return "report"

    if any(x in venue for x in ["journal", "transactions", "magazine"]):
        return "journal-article"

    if any(x in venue for x in ["conference", "proceedings", "symposium", "workshop"]):
        return "proceedings-article"

    return None


In [ ]:
import json
import time
import requests
from pathlib import Path

OPENALEX_RAW_CACHE = Path("data/cache_openalex_raw.json")

def load_cache(path):
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def save_cache(path, cache):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False)

def get_openalex_work(doi):
    if not doi or str(doi).strip() == "":
        return None

    doi = str(doi).strip().lower()

    try:
        r = requests.get(
            f"https://api.openalex.org/works/https://doi.org/{doi}",
            headers={"User-Agent": "DBLP-imputation/1.0"},
            timeout=10
        )
        if r.status_code != 200:
            return None
        return r.json()
    except Exception:
        return None

def extract_abstract_from_work(data):
    if not isinstance(data, dict):
        return None

    inverted_index = data.get("abstract_inverted_index")
    if not inverted_index:
        return None

    max_pos = max((max(pos) for pos in inverted_index.values() if pos), default=-1)
    if max_pos < 0:
        return None

    words = [""] * (max_pos + 1)
    for word, positions in inverted_index.items():
        for pos in positions:
            words[pos] = word

    text = " ".join(words).strip()
    return text if text else None

def extract_references_from_work(data):
    if not isinstance(data, dict):
        return None
    refs = data.get("referenced_works", [])
    return refs if refs else None

def extract_venue_from_work(data):
    if not isinstance(data, dict):
        return None
    venue = data.get("primary_location", {}).get("raw_source_name")
    return venue if venue else None

def extract_doc_type_from_work(data):
    if not isinstance(data, dict):
        return None
    doc_type = data.get("type")
    return doc_type if doc_type else None

def extract_lang_from_work(data):
    if not isinstance(data, dict):
        return None
    lang = data.get("language")
    return lang if lang else None

def extract_keywords_from_work(data):
    if not isinstance(data, dict):
        return None
    concepts = data.get("concepts", [])
    keywords = [
        c["display_name"]
        for c in concepts
        if c.get("level", 0) >= 1 and c.get("score", 0) >= 0.3
    ]
    return keywords if keywords else None

In [ ]:
def impute_from_openalex(df, col, extract_fn, extra_mask=None, sleep_s=0.1):
    raw_cache = load_cache(OPENALEX_RAW_CACHE)

    mask = df[col].apply(is_empty) & ~df["doi"].apply(is_empty)
    if extra_mask is not None:
        mask = mask & extra_mask

    target_idx = df[mask].index.tolist()
    print(f"{col}: rows to inspect = {len(target_idx)}")

    recovered = 0

    for i, idx in enumerate(target_idx, start=1):
        doi = str(df.at[idx, "doi"]).strip().lower()

        if doi in raw_cache:
            data = raw_cache[doi]
        else:
            data = get_openalex_work(doi)
            raw_cache[doi] = data
            if i % 20 == 0:
                save_cache(OPENALEX_RAW_CACHE, raw_cache)
            time.sleep(sleep_s)

        value = extract_fn(data)
        if value is not None and is_empty(df.at[idx, col]):
            df.at[idx, col] = value
            recovered += 1

        if i % 50 == 0:
            print(f"  [{i}/{len(target_idx)}] recovered={recovered}")

    save_cache(OPENALEX_RAW_CACHE, raw_cache)

    print(f"\nImputation completed for {col}")
    print(f"Recovered: {recovered}/{len(target_idx)}")
    print(f"Still missing: {df[col].apply(is_empty).sum()}")

In [ ]:
# import json

# def impute_with_cache(df, col, fetch_fn, cache_path):
#     cache_path = Path(cache_path)
#     mask = df[col].apply(is_empty) & ~df["doi"].apply(is_empty)
#     missing_df = df[mask]

#     if cache_path.exists():
#         with open(cache_path, "r") as f:
#             recovered = {int(k): v for k, v in json.load(f).items()}
#         print(f"Cache found: loaded {len(recovered)} values from {cache_path}")
#     else:
#         recovered = {}
#         for i, (idx, row) in enumerate(missing_df.iterrows()):
#             value = fetch_fn(row["doi"])
#             if value:
#                 recovered[idx] = value

#             if (i + 1) % 10 == 0:
#                 print(f"  [{i+1}/{len(missing_df)}] recovered until now: {len(recovered)}")

#             time.sleep(0.1)

#         with open(cache_path, "w") as f:
#             json.dump(recovered, f)
#         print(f"Cache saved to {cache_path}")

#     for idx, value in recovered.items():
#         df.at[idx, col] = value

#     print(f"\nImputation completed:")
#     print(f"'{col}' records with DOI available: {mask.sum()}")
#     print(f"'{col}' values successfully recovered: {len(recovered)} ({len(recovered)/mask.sum()*100:.1f}%)")
#     missing_total = df[col].apply(is_empty).sum()
#     print(f"'{col}' still missing (total): {missing_total}")

In [ ]:
# def get_lang_from_openalex(doi):
#     if not doi or doi == "":
#         return None
#     try:
#         r = requests.get(
#             f"https://api.openalex.org/works/https://doi.org/{doi}",
#             headers={"User-Agent": "DBLP-imputation/1.0"},
#             timeout=10
#         )
#         if r.status_code != 200:
#             return None
#         return r.json().get("language", None)
#     except Exception:
#         return None


# def get_keywords_from_openalex(doi):
#     if not doi or doi == "":
#         return None
#     try:
#         r = requests.get(
#             f"https://api.openalex.org/works/https://doi.org/{doi}",
#             headers={"User-Agent": "DBLP-imputation/1.0"},
#             timeout=10
#         )
#         if r.status_code != 200:
#             return None
#         concepts = r.json().get("concepts", [])
#         keywords = [
#             c["display_name"] for c in concepts
#             if c.get("level", 0) >= 1 and c.get("score", 0) >= 0.3
#         ]
#         return keywords if keywords else None
#     except Exception:
#         return None
    
# def get_abstract_from_openalex(doi):
#     if not doi or doi == "":
#         return None
#     try:
#         r = requests.get(
#             f"https://api.openalex.org/works/https://doi.org/{doi}",
#             headers={"User-Agent": "DBLP-imputation/1.0"},
#             timeout=10
#         )
#         if r.status_code != 200:
#             return None
        
#         data = r.json()
#         inverted_index = data.get("abstract_inverted_index")
        
#         if not inverted_index:
#             return None
        
#         max_pos = max([max(positions) for positions in inverted_index.values() if positions], default=0)
        
#         if max_pos == 0:
#             return None
        
#         words = [''] * (max_pos + 1)
#         for word, positions in inverted_index.items():
#             for pos in positions:
#                 words[pos] = word
        
#         return ' '.join(words).strip()
#     except Exception:
#         return None
    

# def get_venue_from_openalex(doi):
#     if not doi or doi == "":
#         return None
#     try:
#         r = requests.get(
#             f"https://api.openalex.org/works/https://doi.org/{doi}",
#             headers={"User-Agent": "DBLP-imputation/1.0"},
#             timeout=10
#         )
#         if r.status_code != 200:
#             return None
        
#         data = r.json()
#         venue = data.get("primary_location", {}).get("raw_source_name")
#         return venue if venue else None
#     except Exception:
#         return None
    
# def get_doc_type_from_openalex(doi):
#     if not doi or doi == "":
#         return None
#     try:
#         r = requests.get(
#             f"https://api.openalex.org/works/https://doi.org/{doi}",
#             headers={"User-Agent": "DBLP-imputation/1.0"},
#             timeout=10
#         )
#         if r.status_code != 200:
#             return None
        
#         data = r.json()
#         doc_type = data.get("type")
#         return doc_type if doc_type else None
#     except Exception:
#         return None
    
# def get_references_from_openalex(doi):
#     if not doi or doi == "":
#         return None
#     try:
#         r = requests.get(
#             f"https://api.openalex.org/works/https://doi.org/{doi}",
#             headers={"User-Agent": "DBLP-imputation/1.0"},
#             timeout=10
#         )
#         if r.status_code != 200:
#             return None
        
#         data = r.json()
#         references = data.get("referenced_works", [])
#         return references if references else None
#     except Exception:
#         return None


## 1. Data Creation

###  1.2. Parquet Creation 

da eseguire una sola volta 

In [ ]:
DATA_DIR = Path("data")
SOURCE_PATH = DATA_DIR / "DBLP-Citation-network-V18.jsonl"
TARGET_PATH = DATA_DIR / "DBLP-Citation-network-V18.parquet"
BLOCK_SIZE = 64 * 1024 * 1024  

if not SOURCE_PATH.exists():
    raise FileNotFoundError(f"File non trovato: {SOURCE_PATH}")

if pa.Codec.is_available("zstd"):
    COMPRESSION = "zstd"
elif pa.Codec.is_available("snappy"):
    COMPRESSION = "snappy"
else:
    COMPRESSION = None

print(f"Input : {SOURCE_PATH} ({SOURCE_PATH.stat().st_size / 1024**3:.2f} GiB)")
print(f"Output: {TARGET_PATH}")
print(f"Compressione: {COMPRESSION}")
print(f"Block size: {BLOCK_SIZE / 1024**2:.0f} MiB")

In [ ]:
if TARGET_PATH.exists():
    print(f"Parquet already exists, skipping conversion.")
else:
    reader = paj.open_json(
        SOURCE_PATH,
        read_options=paj.ReadOptions(block_size=BLOCK_SIZE),
    )

    writer = None
    rows_written = 0
    batches_written = 0
    started_at = time.perf_counter()

    try:
        while True:
            try:
                batch = reader.read_next_batch()
            except StopIteration:
                break

            if writer is None:
                writer = pq.ParquetWriter(
                    TARGET_PATH,
                    batch.schema,
                    compression=COMPRESSION,
                )

            writer.write_batch(batch)
            rows_written += batch.num_rows
            batches_written += 1

            if batches_written % 25 == 0:
                elapsed = time.perf_counter() - started_at
                print(f"Batch: {batches_written:>5} | Rows: {rows_written:>12,} | Elapsed: {elapsed:>8.1f}s")

        if writer is None:
            raise RuntimeError("JSONL file seems empty: no batch read.")
    finally:
        reader.close()
        if writer is not None:
            writer.close()

    elapsed = time.perf_counter() - started_at
    print(f"Conversion completed in {elapsed:.1f}s")
    print(f"Rows written : {rows_written:,}")
    print(f"JSONL size   : {SOURCE_PATH.stat().st_size / 1024**3:.2f} GiB")
    print(f"Parquet size : {TARGET_PATH.stat().st_size / 1024**3:.2f} GiB")

### 1.2 Data Creation with DuckDB

Prima di lavorare su batch o sample, facciamo profiling e data quality sull'intero dataset direttamente sul parquet via DuckDB. In questo modo i conteggi dei missing, le distribuzioni per anno e la selezione dei candidati per imputazione sono globali e non dipendono dal sample da 10k record.

In [ ]:
PARQUET_PATH = Path("data/DBLP-Citation-network-V18.parquet")


con = duckdb.connect(":memory:")

# con = duckdb.connect(r"C:\Users\ms\Desktop\dblp.duckdb")

con.execute("PRAGMA threads=4;")
con.execute("PRAGMA enable_progress_bar;")

parquet_str = PARQUET_PATH.as_posix()

con.execute(f"""
    CREATE OR REPLACE VIEW papers AS
    SELECT *
    FROM read_parquet('{parquet_str}');
""")

In [ ]:
display(con.execute("SELECT * FROM papers LIMIT 5").fetchdf())

In [ ]:
# PROFILE_DB_PATH = DATA_DIR / "dblp_profiling.duckdb"

# con_full = duckdb.connect(PROFILE_DB_PATH.as_posix())
# con_full.execute("PRAGMA threads=4;")
# con_full.execute("PRAGMA enable_progress_bar;")

# con_full.execute(f"""
#     CREATE OR REPLACE VIEW papers_full AS
#     SELECT *
#     FROM read_parquet('{parquet_str}');
# """)

# total_rows_full = con_full.execute("SELECT COUNT(*) FROM papers_full").fetchone()[0]
# print(f"Profiling DB: {PROFILE_DB_PATH}")
# print(f"Total rows available in DuckDB: {total_rows_full:,}")

## 2. Data Exploration  

In [ ]:
# pf = pq.ParquetFile(r"data/DBLP-Citation-network-V18.parquet")

# print(f"Total rows   : {pf.metadata.num_rows:,}")
# print(f"Columns      : {pf.metadata.num_columns}")
# print(f"Row groups   : {pf.metadata.num_row_groups}")

In [ ]:
total_rows = con.execute("""
    SELECT COUNT(*)
    FROM papers
""").fetchone()[0]

n_columns = con.execute("""
    DESCRIBE SELECT * FROM papers
""").fetchdf().shape[0]

row_groups = con.execute(f"""
    SELECT COUNT(DISTINCT row_group_id)
    FROM parquet_metadata('{parquet_str}')
""").fetchone()[0]

print(f"Total rows   : {total_rows:,}")
print(f"Columns      : {n_columns}")
print(f"Row groups   : {row_groups}")


In [ ]:
schema_df = con.execute("""
    DESCRIBE SELECT * FROM papers
""").fetchdf()

for i, name in enumerate(schema_df["column_name"]):
    print(f"{i}. {name}")


In [ ]:
schema_df = con.execute("""
    DESCRIBE SELECT * FROM papers
""").fetchdf()

columns = schema_df["column_name"].tolist()

dropdown = widgets.Dropdown(options=columns, description="Colonna:")
output = widgets.Output()

def on_change(change):
    if change["type"] == "change" and change["name"] == "value":
        col = change["new"]
        with output:
            output.clear_output()
            df = con.execute(f'''
                SELECT "{col}"
                FROM papers
                LIMIT 10
            ''').fetchdf()
            display(df)

dropdown.observe(on_change)

display(dropdown, output)

In [ ]:
paper = con.execute("""
    SELECT *
    FROM papers
    LIMIT 1
""").fetchdf().iloc[0]

for col, val in paper.items():
    print(f"{col:15}: {val}")

### 2.1. Data Selection

In [ ]:
con.execute("""
    CREATE OR REPLACE VIEW papers_profile_base AS
    SELECT
        *,
        CASE WHEN title IS NULL OR TRIM(CAST(title AS VARCHAR)) = '' THEN TRUE ELSE FALSE END AS flag_missing_title,
        CASE WHEN abstract IS NULL OR TRIM(CAST(abstract AS VARCHAR)) = '' THEN TRUE ELSE FALSE END AS flag_missing_abstract,
        CASE WHEN doi IS NULL OR TRIM(CAST(doi AS VARCHAR)) = '' THEN TRUE ELSE FALSE END AS flag_missing_doi,
        CASE WHEN lang IS NULL OR TRIM(CAST(lang AS VARCHAR)) = '' THEN TRUE ELSE FALSE END AS flag_missing_lang,
        CASE WHEN venue IS NULL OR TRIM(CAST(venue AS VARCHAR)) = '' THEN TRUE ELSE FALSE END AS flag_missing_venue,
        CASE WHEN doc_type IS NULL OR TRIM(CAST(doc_type AS VARCHAR)) = '' THEN TRUE ELSE FALSE END AS flag_missing_doc_type,
        CASE WHEN COALESCE(array_length(keywords), 0) = 0 THEN TRUE ELSE FALSE END AS flag_missing_keywords,
        CASE WHEN COALESCE(array_length("references"), 0) = 0 THEN TRUE ELSE FALSE END AS flag_missing_references,
        CASE WHEN COALESCE(array_length(authors), 0) = 0 THEN TRUE ELSE FALSE END AS flag_missing_authors,
        CASE WHEN TRY_CAST(year AS INTEGER) IS NULL THEN TRUE ELSE FALSE END AS flag_year_missing,
        TRY_CAST(year AS INTEGER) AS year_int

    FROM papers
""")


### 2.1.1. Missing Values

In [ ]:
missing_summary = con.execute("""
    SELECT * FROM (
        SELECT 'title' AS feature, SUM(CAST(flag_missing_title AS INTEGER)) AS count_rows FROM papers_profile_base
        UNION ALL
        SELECT 'abstract', SUM(CAST(flag_missing_abstract AS INTEGER)) FROM papers_profile_base
        UNION ALL
        SELECT 'doi', SUM(CAST(flag_missing_doi AS INTEGER)) FROM papers_profile_base
        UNION ALL
        SELECT 'lang', SUM(CAST(flag_missing_lang AS INTEGER)) FROM papers_profile_base
        UNION ALL
        SELECT 'venue', SUM(CAST(flag_missing_venue AS INTEGER)) FROM papers_profile_base
        UNION ALL
        SELECT 'doc_type', SUM(CAST(flag_missing_doc_type AS INTEGER)) FROM papers_profile_base
        UNION ALL
        SELECT 'keywords', SUM(CAST(flag_missing_keywords AS INTEGER)) FROM papers_profile_base
        UNION ALL
        SELECT 'references', SUM(CAST(flag_missing_references AS INTEGER)) FROM papers_profile_base
        UNION ALL
        SELECT 'authors', SUM(CAST(flag_missing_authors AS INTEGER)) FROM papers_profile_base
        UNION ALL
        SELECT 'year', SUM(CAST(flag_year_missing AS INTEGER)) FROM papers_profile_base
    )
    ORDER BY count_rows DESC
""").fetchdf()

total_rows = con.execute("SELECT COUNT(*) FROM papers_profile_base").fetchone()[0]
missing_summary["pct_rows"] = missing_summary["count_rows"] / total_rows * 100
missing_summary

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=missing_summary, x="pct_rows", y="feature", color="#C44E52")
plt.title("Missing values by feature")
plt.xlabel("Missing (%)")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

In [ ]:
authors_missing_summary = con.execute("""
WITH authors_exploded AS (
    SELECT
        p.id AS paper_id,
        a.author AS author
    FROM papers_profile_base p
    CROSS JOIN UNNEST(p.authors) AS a(author)
)
SELECT
    COUNT(*) AS total_author_rows,
    SUM(CASE WHEN author.id IS NULL OR TRIM(CAST(author.id AS VARCHAR)) = '' THEN 1 ELSE 0 END) AS missing_author_id,
    SUM(CASE WHEN author.name IS NULL OR TRIM(CAST(author.name AS VARCHAR)) = '' THEN 1 ELSE 0 END) AS missing_author_name,
    SUM(CASE WHEN author.org IS NULL OR TRIM(CAST(author.org AS VARCHAR)) = '' THEN 1 ELSE 0 END) AS missing_author_org
FROM authors_exploded
""").fetchdf()

display(authors_missing_summary)

In [ ]:
missing_by_year = con.execute("""
    SELECT
        year_int AS year,
        COUNT(*) AS n_papers,
        SUM(CAST(flag_missing_abstract AS INTEGER)) AS missing_abstract,
        SUM(CAST(flag_missing_doi AS INTEGER)) AS missing_doi,
        SUM(CAST(flag_missing_venue AS INTEGER)) AS missing_venue,
        SUM(CAST(flag_missing_doc_type AS INTEGER)) AS missing_doc_type,
        SUM(CAST(flag_missing_keywords AS INTEGER)) AS missing_keywords,
        SUM(CAST(flag_missing_references AS INTEGER)) AS missing_references
    FROM papers_profile_base
    WHERE year_int IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").fetchdf()

display(missing_by_year.head(5))

escludiamo direttamente tutti i paper con DOI mancante perche Imputare il DOI partendo da titolo e autori è più complesso per diversi motivi:
Ambiguità dei titoli: Titoli simili o molto generici possono restituire decine di risultati.
Errori nei dati: Un refuso nel nome di un autore o nell'anno di pubblicazione nel tuo dataset può impedire all'algoritmo di trovare il match corretto.
Mancanza del DOI: Non tutti i paper esistenti ne hanno uno (specialmente quelli molto vecchi o di editori minori)

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW papers_scope AS
SELECT *
FROM papers_profile_base
WHERE year_int > 1990
  AND year_int < 2027
  AND NOT flag_missing_doi
""")


In [ ]:
missing_summary_scope = con.execute("""
    SELECT * FROM (
        SELECT 'title' AS feature, SUM(CAST(flag_missing_title AS INTEGER)) AS count_rows FROM papers_scope
        UNION ALL
        SELECT 'abstract', SUM(CAST(flag_missing_abstract AS INTEGER)) FROM papers_scope
        UNION ALL
        SELECT 'lang', SUM(CAST(flag_missing_lang AS INTEGER)) FROM papers_scope
        UNION ALL
        SELECT 'venue', SUM(CAST(flag_missing_venue AS INTEGER)) FROM papers_scope
        UNION ALL
        SELECT 'doc_type', SUM(CAST(flag_missing_doc_type AS INTEGER)) FROM papers_scope
        UNION ALL
        SELECT 'keywords', SUM(CAST(flag_missing_keywords AS INTEGER)) FROM papers_scope
        UNION ALL
        SELECT 'references', SUM(CAST(flag_missing_references AS INTEGER)) FROM papers_scope
        UNION ALL
        SELECT 'authors', SUM(CAST(flag_missing_authors AS INTEGER)) FROM papers_scope
        UNION ALL
        SELECT 'year', SUM(CAST(flag_year_missing AS INTEGER)) FROM papers_scope
    )
    ORDER BY count_rows DESC
""").fetchdf()

total_rows_scope = con.execute("""
SELECT COUNT(*)
FROM papers_scope
""").fetchone()[0]

missing_summary_scope["pct_rows"] = missing_summary_scope["count_rows"] / total_rows_scope * 100
display(missing_summary_scope)

plt.figure(figsize=(10, 6))
sns.barplot(data=missing_summary_scope, x="pct_rows", y="feature", color="#C44E52")
plt.title("Missing values by feature in filtered corpus")
plt.xlabel("Missing (%)")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

In [ ]:
missing_by_year = con.execute("""
    SELECT
        year_int AS year,
        SUM(CAST(flag_missing_abstract AS INTEGER)) AS abstract,
        SUM(CAST(flag_missing_doi AS INTEGER)) AS doi,
        SUM(CAST(flag_missing_venue AS INTEGER)) AS venue,
        SUM(CAST(flag_missing_doc_type AS INTEGER)) AS doc_type,
        SUM(CAST(flag_missing_keywords AS INTEGER)) AS keywords,
        SUM(CAST(flag_missing_references AS INTEGER)) AS "references"
    FROM papers_scope       
    GROUP BY 1
    ORDER BY 1
""").fetchdf()

missing_by_year = missing_by_year.set_index("year")

missing_by_year.plot(
    kind="bar",
    stacked=True,
    figsize=(16, 7),
    colormap="tab20"
)

plt.title("Missing values by year and feature")
plt.xlabel("Year")
plt.ylabel("Missing count")
plt.tight_layout()
plt.show()

In [ ]:
MIN_REFERENCES_RECOVERABLE = 20

con.execute(f"""
CREATE OR REPLACE VIEW papers_record_quality_preimputation AS
SELECT
    *,

    CASE
        WHEN NOT flag_missing_title
         AND NOT flag_year_missing
         AND NOT flag_missing_authors
        THEN TRUE ELSE FALSE
    END AS flag_structurally_valid,

    CASE
        WHEN NOT flag_missing_title
         AND NOT flag_year_missing
         AND NOT flag_missing_authors
         AND NOT flag_missing_abstract
         AND NOT flag_missing_references
        THEN 'good'

        WHEN NOT flag_missing_title
         AND NOT flag_year_missing
         AND NOT flag_missing_authors
         AND flag_missing_abstract
         AND NOT flag_missing_references
         AND NOT flag_missing_doi
         AND COALESCE(array_length("references"), 0) >= {MIN_REFERENCES_RECOVERABLE}
        THEN 'recoverable'

        ELSE 'low_quality'
    END AS paper_segment,

    CASE
        WHEN flag_missing_abstract
         AND NOT flag_missing_references
         AND NOT flag_missing_doi
         AND COALESCE(array_length("references"), 0) >= {MIN_REFERENCES_RECOVERABLE}
        THEN 'high'
        ELSE 'skip'
    END AS openalex_priority
FROM papers_scope
""")


In [ ]:
segment_summary = con.execute(f"""
SELECT
    paper_segment,
    COUNT(*) AS n_papers,
    SUM(CASE WHEN NOT flag_missing_abstract THEN 1 ELSE 0 END) AS has_abstract,
    SUM(CASE WHEN NOT flag_missing_references THEN 1 ELSE 0 END) AS has_references,
    SUM(CASE WHEN NOT flag_missing_doi THEN 1 ELSE 0 END) AS has_doi,
    SUM(CASE WHEN COALESCE(array_length("references"), 0) >= {MIN_REFERENCES_RECOVERABLE} THEN 1 ELSE 0 END) AS refs_ge_threshold
FROM papers_record_quality_preimputation
GROUP BY 1
""").fetchdf()

In [ ]:
total_rows_pre = con.execute("""
SELECT COUNT(*)
FROM papers_record_quality_preimputation
""").fetchone()[0]


segment_order = ["good", "recoverable", "low_quality"]
segment_summary["paper_segment"] = pd.Categorical(
    segment_summary["paper_segment"],
    categories=segment_order,
    ordered=True
)
segment_summary = segment_summary.sort_values("paper_segment")
segment_summary["pct_papers"] = segment_summary["n_papers"] / total_rows_pre * 100

display(segment_summary)


we keep only good and recoverable

In [ ]:
df = con.execute("""
    SELECT p.*
    FROM papers AS p
    WHERE EXISTS (
        SELECT 1
        FROM papers_record_quality_preimputation q
        WHERE q.id = p.id
          AND q.paper_segment IN ('good', 'recoverable')
    )
""").fetchdf()

In [ ]:
df.shape

spiegare che ci interessa imputare solo le variabili che pensiamo possano essere utili per predirre il numero di citazioni o se un paper cita un'altro, ma prima potrebbe essere utile analizzare la distribuzione nei anni dei missing value.

#### 1. Abstract imputation

In [ ]:
impute_from_openalex(df, "abstract", extract_abstract_from_work)

#### 2. Lang imputation

spiegare che se avessivo avuto accesso ha una parte del paper avremmo potuto usare un modello dei trasformer per analizzare la scrittura del paper e imputare in modo corretto la lingua del paper, siccome non è il caso possiamo provare altri metodi. 

Prima di tutto dobbiamo vedere le possibilità o comunque i valori che abbiamo presenti, in caso fossero tutti inglesi allora li mettiamo inglesi tutti. 

In [ ]:
lang_counts = df["lang"].value_counts(dropna=False)
lang_counts.index = lang_counts.index.fillna("(null)")
print(f"Unique values of 'lang' (sample of {n:,} rows):\n")
print(lang_counts.to_string())

spiegare che come vediamo abbiamo diverse lingue nel dataset, quindi decidiamo di usare un modello (spiegare il modello)

In [ ]:
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

SUPPORTED_LANGS = {"en", "de", "fr", "pt", "es", "it", "zh", "ja", "ko", "ru"}

def detect_lang(row):
    parts = []
    for col in ["title", "abstract", "venue"]:
        val = row.get(col)
        if isinstance(val, str) and val.strip():
            parts.append(val.strip())
    
    text = " ".join(parts)
    
    if len(text) < 30:
        return None
    
    try:
        pred = detect(text)
        return pred if pred in SUPPORTED_LANGS else None
    except:
        return None

missing_lang_mask = df["lang"].apply(is_empty)

df.loc[missing_lang_mask, "lang"] = (
    df.loc[missing_lang_mask]
      .apply(detect_lang, axis=1)
)

In [ ]:
lang_counts = df["lang"].value_counts(dropna=False)
lang_counts.index = [
    "(null)" if pd.isna(x) or x == "" else x
    for x in lang_counts.index
]

print(f"Unique values of 'lang' (sample of {n:,} rows):\n")
print(lang_counts.to_string())


come possiamo vedere non abbiamo più valori nan 

#### 3. Keywords imputation

come primo tentativo possiamo vedere se con il doi e usando onealex possiamo ottenere delle keywords 

In [ ]:
import yake

YAKE_LANG_MAP = {
    "en": "en",
    "it": "it",
    "fr": "fr",
    "de": "de",
    "es": "es",
    "pt": "pt",
    "zh": "zh",
    "ja": "ja",
    "ko": "ko",
    "ru": "ru",
}

def extract_keywords_local(row):
    if not is_empty(row.get("keywords")):
        return row.get("keywords")

    parts = []
    for col in ["title", "abstract"]:
        val = row.get(col)
        if isinstance(val, str) and val.strip():
            parts.append(val.strip())

    text = " ".join(parts)
    if len(text) < 40:
        return None

    lang = row.get("lang")
    yake_lang = YAKE_LANG_MAP.get(lang, "en")

    try:
        extractor = yake.KeywordExtractor(
            lan=yake_lang,
            n=2,
            top=8,
            dedupLim=0.9
        )
        kws = extractor.extract_keywords(text)
        keywords = [kw for kw, score in kws]
        return keywords if keywords else None
    except Exception:
        return None


In [ ]:
missing_keywords_mask = df["keywords"].apply(is_empty)

df.loc[missing_keywords_mask, "keywords"] = (
    df.loc[missing_keywords_mask]
      .apply(extract_keywords_local, axis=1)
)


In [ ]:
#impute_with_cache(df, "keywords", get_keywords_from_openalex, "data/cache_keywords.json")

siamo riusciti ad imputare un totale di circa 951 valori, i rimanenti sono dovuti a richieste mancanti di DOI, per il resto possiamo generarli partendo dall'absatract

In [ ]:
# CACHE_PATH = Path("data/cache_keywords_transformer.json")

# tokenizer = AutoTokenizer.from_pretrained("fabiochiu/t5-base-tag-generation")
# model = AutoModelForSeq2SeqLM.from_pretrained("fabiochiu/t5-base-tag-generation")

# def extract_keywords_from_abstract(abstract):
#     if not abstract or abstract == "":
#         return None
#     try:
#         inputs = tokenizer.encode(abstract[:512], return_tensors="pt", max_length=512, truncation=True)
#         outputs = model.generate(inputs, max_length=50, num_beams=4, early_stopping=True)
#         result = tokenizer.decode(outputs[0], skip_special_tokens=True)
#         keywords = [kw.strip() for kw in result.split(",") if kw.strip()]
#         return keywords if keywords else None
#     except Exception as e:
#         print(f"Error processing abstract: {e}")
#         return None

# mask = df["keywords"].apply(is_empty) & ~df["abstract"].apply(is_empty)
# print(f"Records to impute with transformer: {mask.sum()}")

# # Load cache if exists
# if CACHE_PATH.exists():
#     with open(CACHE_PATH, "r") as f:
#         recovered = {int(k): v for k, v in json.load(f).items()}
#     print(f"Cache found: loaded {len(recovered)} values")
#     for idx, keywords in recovered.items():
#         df.at[idx, "keywords"] = keywords
# else:
#     recovered = {}
#     for i, (idx, row) in enumerate(df[mask].iterrows()):
#         keywords = extract_keywords_from_abstract(row["abstract"])
#         if keywords:
#             df.at[idx, "keywords"] = keywords
#             recovered[idx] = keywords
#         if (i + 1) % 10 == 0:
#             print(f"  [{i+1}/{mask.sum()}] recovered so far: {len(recovered)}")

#     with open(CACHE_PATH, "w") as f:
#         json.dump(recovered, f)
#     print(f"Cache saved to {CACHE_PATH}")

# print(f"Still missing: {df['keywords'].apply(is_empty).sum()}")

come possiamo notare i rimanenti sono dovuti alla mancanza di un abstract il quale non ci permette di fare l'imputazione 

#### 4. Venue

In [ ]:
# impute_with_cache(df, "venue", get_venue_from_openalex, "data/cache_venue.json")

In [ ]:
need_openalex = df["abstract"].apply(is_empty) & ~df["doi"].apply(is_empty)

impute_from_openalex(
    df,
    "venue",
    extract_venue_from_work,
    extra_mask=need_openalex
)


#### 5. Doc_type

In [ ]:
# impute_with_cache(df, "doc_type", get_doc_type_from_openalex, "data/cache_doc_type.json")

In [ ]:
missing_doc_type_mask = df["doc_type"].apply(is_empty)
df.loc[missing_doc_type_mask, "doc_type"] = (
    df.loc[missing_doc_type_mask]
      .apply(infer_doc_type_local, axis=1)
)

#### 6. Reverence

In [ ]:
# impute_with_cache(df, "references", get_references_from_openalex, "data/cache_references.json")

In [ ]:
impute_from_openalex(
    df,
    "references",
    extract_references_from_work
)


#### 7. Authors.id

In [ ]:
def impute_authors_id(df):
    """
    Imputa authors.id cercando lo stesso nome in altri paper
    Se trova lo stesso nome, usa l'id già assegnato
    """
    
    # Crea un mapping: nome -> id (da paper dove abbiamo l'id)
    name_to_id = {}
    
    for idx, row in df.iterrows():
        authors = row.get("authors")
        if authors is not None and len(authors) > 0:  # Cambia qui
            for author in authors:
                if isinstance(author, dict):
                    name = author.get("name")
                    author_id = author.get("id")
                    # Se abbiamo sia nome che id, salvalo nel mapping
                    if name and author_id:
                        if name not in name_to_id:
                            name_to_id[name] = author_id
    
    print(f"Created mapping for {len(name_to_id):,} unique author names")
    
    # Ora imputa gli id mancanti usando il mapping
    recovered = 0
    for idx, row in df.iterrows():
        authors = row.get("authors")
        if authors is not None and len(authors) > 0:  # Cambia qui
            for author in authors:
                if isinstance(author, dict):
                    name = author.get("name")
                    author_id = author.get("id")
                    # Se manca l'id ma abbiamo il nome, usa il mapping
                    if not author_id and name and name in name_to_id:
                        author["id"] = name_to_id[name]
                        recovered += 1
    
    print(f"Recovered {recovered:,} author IDs")
    return df

In [ ]:
df = impute_authors_id(df)

#### 8. Authors.org

In [ ]:
def impute_authors_org(df):
    """
    Imputa authors.org usando il paper più vicino temporalmente (±365 giorni)
    dello stesso autore
    """
    
    # Mapping: (author_name, year) -> org (da paper dove abbiamo org)
    author_year_to_org = {}
    
    # Prima pass: costruisci il mapping
    for idx, row in df.iterrows():
        year = row.get("year")
        authors = row.get("authors")
        
        if year is not None and authors is not None and len(authors) > 0:  # Cambia qui
            for author in authors:
                if isinstance(author, dict):
                    name = author.get("name")
                    org = author.get("org")
                    # Se abbiamo nome, anno e org, salvalo
                    if name and org:
                        try:
                            key = (name, int(year))
                            if key not in author_year_to_org:
                                author_year_to_org[key] = org
                        except:
                            pass
    
    print(f"Created mapping for {len(author_year_to_org):,} (author, year) pairs")
    
    # Seconda pass: imputa gli org mancanti
    recovered = 0
    for idx, row in df.iterrows():
        year = row.get("year")
        authors = row.get("authors")
        
        if year is not None and authors is not None and len(authors) > 0:  # Cambia qui
            try:
                year = int(year)
                for author in authors:
                    if isinstance(author, dict):
                        name = author.get("name")
                        org = author.get("org")
                        
                        # Se manca org ma abbiamo nome e anno
                        if not org and name:
                            # Cerca nel range ±365 giorni
                            best_org = None
                            best_distance = float('inf')
                            
                            for (author_name, author_year), author_org in author_year_to_org.items():
                                if author_name == name:
                                    distance = abs(author_year - year)
                                    if distance <= 365 and distance < best_distance:
                                        best_org = author_org
                                        best_distance = distance
                            
                            if best_org:
                                author["org"] = best_org
                                recovered += 1
            except:
                pass
    
    print(f"Recovered {recovered:,} author ORGs")
    return df

In [ ]:
# Esegui
df = impute_authors_org(df)

#### Comparison missing values 

In [ ]:
missing_count_after = {}

# Simple columns
for col in ["id", "title", "abstract", "year", "page_start", "page_end",
            "lang", "volume", "issue", "issn", "isbn", "doi", "venue", "doc_type"]:
    missing_count_after[col] = df[col].apply(is_empty).sum()

# Lists
for col in ["keywords", "references", "url"]:
    missing_count_after[col] = df[col].apply(is_empty).sum()

# Authors AFTER
authors_flat_after = pd.DataFrame(df["authors"].explode().dropna().tolist())
missing_authors_after = {}
for col in ["id", "org"]:
    missing = authors_flat_after[col].apply(is_empty).sum()
    missing_authors_after[f"authors.{col}"] = missing

missing_count_after["authors.id"] = missing_authors_after["authors.id"]
missing_count_after["authors.org"] = missing_authors_after["authors.org"]

In [ ]:
import plotly.graph_objects as go

imputated_cols = ["abstract", "lang", "keywords", "venue", "doc_type", "references", "authors.id", "authors.org"]  

cols_to_plot = []
before = []
after = []

for col in imputated_cols:
    if col in missing_count_before.keys():
        cols_to_plot.append(col)
        before.append(missing_count_before[col])
        after.append(missing_count_after[col])
    elif col in missing_authors_before.keys():
        cols_to_plot.append(col)
        before.append(missing_authors_before[col])
        after.append(missing_authors_after[col])

improvement = [(b - a) for b, a in zip(before, after)]
improvement_pct = [((b - a) / b * 100) if b > 0 else 0 for b, a in zip(before, after)]

fig = go.Figure(data=[
    go.Bar(name='Before', x=cols_to_plot, y=before, marker_color='#E45756'),
    go.Bar(name='After', x=cols_to_plot, y=after, marker_color='#54A24B'),
    go.Bar(name='Recovered', x=cols_to_plot, y=improvement, marker_color='#4C72B0')
])

fig.update_layout(
    title="Missing Values: Before vs After Imputation (All Imputated Columns)",
    xaxis_title="Column",
    yaxis_title="Missing Count",
    barmode='group',
    height=600,
    template='plotly_white',
    hovermode='x unified',
    font=dict(size=12)
)

fig.show()

## Data Visualizzation 

In [ ]:
year_counts = (
    df["year"]
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
)

year_counts = year_counts.reindex(
    range(year_counts.index.min(), year_counts.index.max() + 1),
    fill_value=0
)
rolling_avg = year_counts.rolling(window=5, center=True, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(15, 6))
ax.bar(
    year_counts.index,
    year_counts.values,
    width=0.85,
    color="#4C78A8",
    edgecolor="#1F2A44",
    linewidth=0.6,
    alpha=0.9,
    label="Papers per year"
)

peak_year = int(year_counts.idxmax())
peak_value = int(year_counts.max())
ax.scatter([peak_year], [peak_value], color="#E45756", s=45, zorder=3)
ax.annotate(
    f"Peak: {peak_value} papers ({peak_year})",
    xy=(peak_year, peak_value),
    xytext=(peak_year - 12, peak_value + 35),
    arrowprops=dict(arrowstyle="->", color="#444444", lw=1),
    fontsize=10
)

tick_years = list(range(year_counts.index.min(), year_counts.index.max() + 1, 5))
if year_counts.index.max() not in tick_years:
    tick_years.append(year_counts.index.max())

ax.set_xticks(tick_years)
ax.set_xlim(year_counts.index.min() - 1, year_counts.index.max() + 1)
ax.set_title("Distribution of papers by publication year", pad=12)
ax.set_xlabel("Publication year")
ax.set_ylabel("Number of papers")
ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.legend(frameon=False)
sns.despine(ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
tmp = (
    df.dropna(subset=["year"])
      .assign(year=lambda x: x["year"].astype(int))
      .groupby(["year", "doc_type"])
      .size()
      .unstack(fill_value=0)
)

tmp.plot.area(figsize=(14, 6), alpha=0.8)
plt.title("Document types over time")
plt.xlabel("Year")
plt.ylabel("Number of papers")
plt.tight_layout()
plt.show()


In [ ]:
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 5)

def is_empty(x):
    if x is None:
        return True
    if isinstance(x, str):
        return x.strip() == ""
    if isinstance(x, (list, tuple, set, dict)):
        return len(x) == 0
    if isinstance(x, np.ndarray):
        return x.size == 0
    return pd.isna(x)

In [ ]:
cols_to_check = ["id", "title", "abstract", "year", "lang", "venue", "doc_type",
                 "keywords", "references", "doi", "issn", "isbn", "authors"]

missing_summary = pd.DataFrame({
    "missing_count": [df[col].apply(is_empty).sum() for col in cols_to_check],
    "missing_pct": [df[col].apply(is_empty).mean() * 100 for col in cols_to_check]
}, index=cols_to_check).sort_values("missing_pct", ascending=False)

display(missing_summary)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=missing_summary.reset_index(),
    x="missing_pct",
    y="index",
    color="#E45756"
)
plt.title("Missing values by column")
plt.xlabel("Missing (%)")
plt.ylabel("Column")
plt.tight_layout()
plt.show()


In [ ]:
important_cols = ["abstract", "doi", "venue", "lang", "keywords", "references"]

heatmap_df = pd.DataFrame(index=sorted(df["year"].dropna().astype(int).unique()))

for col in important_cols:
    tmp = (
        df.dropna(subset=["year"])
          .assign(year=lambda x: x["year"].astype(int),
                  missing=lambda x: x[col].apply(is_empty))
          .groupby("year")["missing"]
          .mean()
          .mul(100)
    )
    heatmap_df[col] = tmp

plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_df, cmap="Reds", annot=False)
plt.title("Missing percentage by year and variable")
plt.xlabel("Variable")
plt.ylabel("Year")
plt.tight_layout()
plt.show()

In [ ]:
cols = ["abstract", "doi", "venue", "lang", "keywords", "references"]

missing_flags = pd.DataFrame({
    col: df[col].apply(is_empty).astype(int)
    for col in cols
})

corr = missing_flags.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation of missingness")
plt.tight_layout()
plt.show()


matrice interessante per capire su quali features "usare" per fare filling di alcune missing

In [ ]:
important_cols = ["abstract", "doi", "venue", "lang", "keywords", "references"]

def plot_missing_by_year(col):
    missing_by_year = (
        df.dropna(subset=["year"])
          .assign(
              year=lambda x: x["year"].astype(int),
              missing=lambda x: x[col].apply(is_empty)
          )
          .groupby("year")["missing"]
          .agg(["sum", "mean", "count"])
          .reset_index()
    )

    missing_by_year["mean"] *= 100

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    sns.lineplot(
        data=missing_by_year,
        x="year",
        y="sum",
        ax=axes[0],
        marker="o"
    )
    axes[0].set_title(f"Missing {col} by year")
    axes[0].set_xlabel("Year")
    axes[0].set_ylabel("Missing count")

    sns.lineplot(
        data=missing_by_year,
        x="year",
        y="mean",
        ax=axes[1],
        marker="o",
        color="#E45756"
    )
    axes[1].set_title(f"Missing {col} by year (%)")
    axes[1].set_xlabel("Year")
    axes[1].set_ylabel("Missing percentage")

    plt.tight_layout()
    plt.show()

widgets.interact(plot_missing_by_year, col=important_cols)

plot per capire una prima similarità tra documenti

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
import plotly.express as px

text = (df["title"].fillna("") + " " + df["abstract"].fillna("")).fillna("")

vectorizer = TfidfVectorizer(max_features=2000, stop_words="english")
X = vectorizer.fit_transform(text)

coords = PCA(n_components=2).fit_transform(X.toarray())

plot_df = df.copy()
plot_df["x"] = coords[:, 0]
plot_df["y"] = coords[:, 1]

fig = px.scatter(
    plot_df,
    x="x",
    y="y",
    color="lang",
    hover_data=["title", "year", "venue"],
    title="Document map"
)
fig.show()

## Data Quality and Normalization

qui dobbiamo normalizzare testo, DOI, venue, doc_type e nomi autore, creare flag di qualità dati, evidenziare inconsistenze utili per il task finale

In [ ]:
df_clean = df.copy()

In [ ]:
import re
import math
import unicodedata
from collections import defaultdict

CURRENT_YEAR = 2026
MIN_REASONABLE_YEAR = 1950

ZERO_WIDTH_PATTERN = re.compile(r"[\u200b-\u200d\ufeff]")
MULTISPACE_PATTERN = re.compile(r"\s+")
PUNCT_TO_SPACE_PATTERN = re.compile(r"[^a-z0-9]+")
DOI_PREFIX_PATTERN = re.compile(r"^(https?://(dx\.)?doi\.org/|doi:)", re.IGNORECASE)

VENUE_REPLACEMENTS = {
    "intl": "international",
    "int'l": "international",
    "conf": "conference",
    "proc": "proceedings",
    "symp": "symposium",
    "sympos": "symposium",
    "worksh": "workshop",
    "trans": "transactions",
    "j": "journal",
}

DOC_TYPE_RULES = [
    ("conference", ("conference", "proceedings", "symposium", "workshop")),
    ("journal", ("journal", "transactions", "magazine")),
    ("book-chapter", ("chapter", "book chapter")),
    ("book", ("book",)),
    ("preprint", ("preprint", "posted content")),
    ("dissertation", ("dissertation", "thesis")),
    ("report", ("report", "technical report")),
]

def is_empty(x):
    if x is None:
        return True
    if isinstance(x, float) and math.isnan(x):
        return True
    if isinstance(x, str):
        return x.strip() == ""
    if isinstance(x, np.ndarray):
        return x.size == 0
    if isinstance(x, (list, tuple, set, dict)):
        return len(x) == 0
    return False

def ensure_list(x):
    if x is None:
        return []
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, list):
        return x
    if isinstance(x, tuple):
        return list(x)
    return []

def clean_string(x):
    if x is None:
        return ""
    if isinstance(x, float) and math.isnan(x):
        return ""
    return str(x).strip()

def normalize_text(x, lowercase=True):
    text = clean_string(x)
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = ZERO_WIDTH_PATTERN.sub("", text)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text.lower() if lowercase else text

def strip_accents(x):
    text = normalize_text(x, lowercase=True)
    if not text:
        return ""
    decomposed = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in decomposed if not unicodedata.combining(ch))

def normalize_for_key(x):
    text = strip_accents(x)
    text = PUNCT_TO_SPACE_PATTERN.sub(" ", text)
    return MULTISPACE_PATTERN.sub(" ", text).strip()

def normalize_doi(x):
    doi = normalize_text(x, lowercase=True)
    if not doi:
        return ""
    doi = DOI_PREFIX_PATTERN.sub("", doi)
    return doi.strip().rstrip("/")

def normalize_author_name(x):
    name = normalize_text(x, lowercase=False)
    if not name:
        return ""

    if "," in name:
        left, right = [part.strip() for part in name.split(",", 1)]
        if left and right:
            name = f"{right} {left}"

    name = normalize_text(name, lowercase=True)
    name = re.sub(r"[.'`]", "", name)
    name = re.sub(r"[^a-z0-9\s\-]", " ", strip_accents(name))
    return MULTISPACE_PATTERN.sub(" ", name).strip()

def normalize_venue(x):
    venue = normalize_text(x, lowercase=True)
    if not venue:
        return ""

    venue = venue.replace("&", " and ")
    venue = re.sub(r"[./,;:()\-]+", " ", venue)

    tokens = []
    for token in venue.split():
        token = token.strip(".")
        tokens.append(VENUE_REPLACEMENTS.get(token, token))

    venue = " ".join(tokens)
    venue = strip_accents(venue)
    venue = re.sub(r"\bproc of the\b", "proceedings of", venue)
    venue = re.sub(r"\bproc\b", "proceedings", venue)
    return MULTISPACE_PATTERN.sub(" ", venue).strip()

def normalize_doc_type(x):
    text = normalize_text(x, lowercase=True)
    if not text:
        return ""
    for canonical, cues in DOC_TYPE_RULES:
        if any(cue in text for cue in cues):
            return canonical
    return text

def normalize_lang(x):
    lang = normalize_text(x, lowercase=True)
    return lang[:5] if lang else ""

def contains_non_ascii(text):
    return any(ord(ch) > 127 for ch in text)


In [ ]:
df_clean["title_clean"] = df_clean["title"].apply(lambda x: normalize_text(x, lowercase=False))
df_clean["title_norm"] = df_clean["title"].apply(normalize_for_key)

df_clean["abstract_clean"] = df_clean["abstract"].apply(lambda x: normalize_text(x, lowercase=False))
df_clean["abstract_norm"] = df_clean["abstract"].apply(normalize_for_key)

df_clean["venue_clean"] = df_clean["venue"].apply(lambda x: normalize_text(x, lowercase=False))
df_clean["venue_norm"] = df_clean["venue"].apply(normalize_venue)

df_clean["doc_type_clean"] = df_clean["doc_type"].apply(lambda x: normalize_text(x, lowercase=False))
df_clean["doc_type_norm"] = df_clean["doc_type"].apply(normalize_doc_type)

df_clean["doi_norm"] = df_clean["doi"].apply(normalize_doi)
df_clean["lang_norm"] = df_clean["lang"].apply(normalize_lang)
df_clean["year_clean"] = pd.to_numeric(df_clean["year"], errors="coerce")

df_clean["keywords_norm"] = df_clean["keywords"].apply(
    lambda xs: list(dict.fromkeys([normalize_for_key(x) for x in ensure_list(xs) if normalize_for_key(x)]))
)
df_clean["n_keywords"] = df_clean["keywords_norm"].apply(len)
df_clean["n_references"] = df_clean["references"].apply(lambda x: len(ensure_list(x)))


In [ ]:
cols_preview = [
    "title", "title_norm",
    "venue", "venue_norm",
    "doc_type", "doc_type_norm",
    "doi", "doi_norm",
    "lang", "lang_norm"
]
display(df_clean[cols_preview].head(10))


In [ ]:
def extract_author_features(authors):
    authors = ensure_list(authors)

    author_names_raw = []
    author_names_norm = []
    author_names_ascii = []
    author_ids_clean = []
    author_orgs_clean = []
    author_missing_id_count = 0
    author_missing_org_count = 0
    author_non_ascii_count = 0

    pairs = []

    for author in authors:
        if not isinstance(author, dict):
            continue

        raw_name = normalize_text(author.get("name"), lowercase=False)
        norm_name = normalize_author_name(author.get("name"))
        ascii_name = normalize_for_key(author.get("name"))
        author_id = normalize_text(author.get("id"), lowercase=False)
        author_org = normalize_text(author.get("org"), lowercase=False)

        if raw_name:
            author_names_raw.append(raw_name)
        if norm_name:
            author_names_norm.append(norm_name)
        if ascii_name:
            author_names_ascii.append(ascii_name)
        if raw_name and ascii_name:
            pairs.append((raw_name, ascii_name))

        if author_id:
            author_ids_clean.append(author_id)
        else:
            author_missing_id_count += 1

        if author_org:
            author_orgs_clean.append(author_org)
        else:
            author_missing_org_count += 1

        if raw_name and contains_non_ascii(raw_name):
            author_non_ascii_count += 1

    seen = set()
    unique_pairs = []
    for pair in pairs:
        if pair not in seen:
            seen.add(pair)
            unique_pairs.append(pair)

    raw_by_ascii = defaultdict(set)
    for raw_name, ascii_name in unique_pairs:
        raw_by_ascii[ascii_name].add(raw_name)

    flag_author_name_variants_in_paper = any(len(v) > 1 for v in raw_by_ascii.values())

    return pd.Series({
        "author_names_raw": [raw for raw, _ in unique_pairs],
        "author_names_norm": list(dict.fromkeys(author_names_norm)),
        "author_names_ascii": [ascii_name for _, ascii_name in unique_pairs],
        "author_ids_clean": list(dict.fromkeys(author_ids_clean)),
        "author_orgs_clean": list(dict.fromkeys(author_orgs_clean)),
        "n_authors": len(author_names_raw),
        "author_missing_id_count": author_missing_id_count,
        "author_missing_org_count": author_missing_org_count,
        "author_non_ascii_count": author_non_ascii_count,
        "flag_author_name_variants_in_paper": flag_author_name_variants_in_paper
    })

author_features = df_clean["authors"].apply(extract_author_features)
df_clean = pd.concat([df_clean, author_features], axis=1)


In [ ]:
display(
    df_clean[
        [
            "authors",
            "author_names_raw",
            "author_names_norm",
            "author_names_ascii",
            "n_authors",
            "author_missing_id_count",
            "author_missing_org_count"
        ]
    ].head(5)
)


In [ ]:
def venue_doc_type_conflict(venue_norm, doc_type_norm):
    if not venue_norm or not doc_type_norm:
        return False

    venue_looks_conference = any(token in venue_norm for token in ["conference", "proceedings", "symposium", "workshop"])
    venue_looks_journal = any(token in venue_norm for token in ["journal", "transactions", "magazine"])

    return (
        (venue_looks_conference and doc_type_norm == "journal") or
        (venue_looks_journal and doc_type_norm == "conference")
    )

df_clean["flag_missing_title"] = df_clean["title_clean"].eq("")
df_clean["flag_missing_abstract"] = df_clean["abstract_clean"].eq("")
df_clean["flag_missing_venue"] = df_clean["venue_clean"].eq("")
df_clean["flag_missing_doc_type"] = df_clean["doc_type_clean"].eq("")
df_clean["flag_missing_doi"] = df_clean["doi_norm"].eq("")
df_clean["flag_missing_keywords"] = df_clean["n_keywords"].eq(0)
df_clean["flag_missing_references"] = df_clean["n_references"].eq(0)
df_clean["flag_missing_authors"] = df_clean["n_authors"].eq(0)

df_clean["flag_author_id_missing_any"] = df_clean["author_missing_id_count"].gt(0)
df_clean["flag_author_org_missing_any"] = df_clean["author_missing_org_count"].gt(0)
df_clean["flag_author_non_ascii_present"] = df_clean["author_non_ascii_count"].gt(0)

df_clean["flag_year_missing"] = df_clean["year_clean"].isna()
df_clean["flag_year_suspicious"] = df_clean["year_clean"].apply(
    lambda x: False if pd.isna(x) else int(x) < MIN_REASONABLE_YEAR or int(x) > CURRENT_YEAR + 1
)
df_clean["flag_doc_type_unknown"] = df_clean["doc_type_norm"].eq("")
df_clean["flag_venue_doc_type_conflict"] = [
    venue_doc_type_conflict(v, d)
    for v, d in zip(df_clean["venue_norm"], df_clean["doc_type_norm"])
]

quality_cols = [
    "flag_missing_title",
    "flag_missing_abstract",
    "flag_missing_venue",
    "flag_missing_doc_type",
    "flag_missing_doi",
    "flag_missing_keywords",
    "flag_missing_references",
    "flag_missing_authors",
    "flag_author_id_missing_any",
    "flag_author_org_missing_any",
    "flag_year_missing",
    "flag_year_suspicious",
    "flag_venue_doc_type_conflict"
]

df_clean["quality_score"] = df_clean[quality_cols].sum(axis=1).astype(int)


In [ ]:
quality_summary = pd.DataFrame({
    "count": df_clean[quality_cols].sum().astype(int),
    "pct": (df_clean[quality_cols].mean() * 100).round(2)
}).sort_values("count", ascending=False)

display(quality_summary)


In [ ]:
plot_df = quality_summary.reset_index().rename(columns={"index": "flag"})

plt.figure(figsize=(12, 7))
sns.barplot(data=plot_df, x="count", y="flag", color="#D95F02")
plt.title("Data quality flags")
plt.xlabel("Rows flagged")
plt.ylabel("")
plt.tight_layout()
plt.show()


In [ ]:
score_dist = (
    df_clean["quality_score"]
    .value_counts()
    .sort_index()
    .rename_axis("quality_score")
    .reset_index(name="count")
)

plt.figure(figsize=(10, 6))
sns.barplot(data=score_dist, x="quality_score", y="count", color="#1B9E77")
plt.title("Quality score distribution")
plt.xlabel("Quality score")
plt.ylabel("Rows")
plt.tight_layout()
plt.show()


In [ ]:
year_quality = (
    df_clean.dropna(subset=["year_clean"])
    .assign(year_clean=lambda x: x["year_clean"].astype(int))
    .groupby("year_clean")[[
        "flag_missing_abstract",
        "flag_author_org_missing_any",
        "flag_venue_doc_type_conflict"
    ]]
    .mean()
    .mul(100)
    .reset_index()
)

year_quality_long = year_quality.melt(id_vars="year_clean", var_name="issue", value_name="pct")

plt.figure(figsize=(14, 6))
sns.lineplot(data=year_quality_long, x="year_clean", y="pct", hue="issue", marker="o")
plt.title("Quality issues by year")
plt.xlabel("Year")
plt.ylabel("Rows flagged (%)")
plt.tight_layout()
plt.show()


In [ ]:
author_alias_rows = []

for _, row in df_clean.iterrows():
    raw_names = row["author_names_raw"]
    ascii_names = row["author_names_ascii"]

    for raw_name, ascii_name in zip(raw_names, ascii_names):
        author_alias_rows.append({
            "author_ascii": ascii_name,
            "author_raw": raw_name
        })

author_alias_df = pd.DataFrame(author_alias_rows)

author_alias_summary = (
    author_alias_df.groupby("author_ascii")
    .agg(
        n_variants=("author_raw", "nunique"),
        examples=("author_raw", lambda x: " | ".join(sorted(set(list(x)[:10]))[:5]))
    )
    .reset_index()
    .sort_values(["n_variants", "author_ascii"], ascending=[False, True])
)

author_alias_summary = author_alias_summary[author_alias_summary["n_variants"] > 1]
display(author_alias_summary.head(20))


In [ ]:
top_author_alias = author_alias_summary.head(15).sort_values("n_variants", ascending=True)

plt.figure(figsize=(12, 7))
sns.barplot(data=top_author_alias, x="n_variants", y="author_ascii", color="#7570B3")
plt.title("Author aliases collapsed by normalization")
plt.xlabel("Distinct raw spellings")
plt.ylabel("Author normalized key")
plt.tight_layout()
plt.show()


In [ ]:
venue_alias_df = (
    df_clean.loc[df_clean["venue_norm"] != "", ["venue_clean", "venue_norm"]]
    .drop_duplicates()
)

venue_alias_summary = (
    venue_alias_df.groupby("venue_norm")
    .agg(
        n_variants=("venue_clean", "nunique"),
        examples=("venue_clean", lambda x: " | ".join(sorted(set(list(x)[:10]))[:5]))
    )
    .reset_index()
    .sort_values(["n_variants", "venue_norm"], ascending=[False, True])
)

venue_alias_summary = venue_alias_summary[venue_alias_summary["n_variants"] > 1]
display(venue_alias_summary.head(20))


In [ ]:
top_venue_alias = venue_alias_summary.head(15).sort_values("n_variants", ascending=True)

plt.figure(figsize=(12, 7))
sns.barplot(data=top_venue_alias, x="n_variants", y="venue_norm", color="#E7298A")
plt.title("Venue aliases collapsed by normalization")
plt.xlabel("Distinct raw spellings")
plt.ylabel("Venue normalized key")
plt.tight_layout()
plt.show()


In [ ]:
conflict_examples = df_clean.loc[
    df_clean["flag_venue_doc_type_conflict"],
    ["id", "title", "year_clean", "venue_clean", "venue_norm", "doc_type_clean", "doc_type_norm"]
].copy()

print(f"Numero di conflitti venue/doc_type: {len(conflict_examples):,}")
display(conflict_examples.head(20))


In [ ]:
author_problem_examples = df_clean.loc[
    df_clean["flag_author_id_missing_any"] | df_clean["flag_author_org_missing_any"] | df_clean["flag_author_name_variants_in_paper"],
    [
        "id",
        "title",
        "year_clean",
        "author_names_raw",
        "author_names_norm",
        "author_ids_clean",
        "author_orgs_clean",
        "author_missing_id_count",
        "author_missing_org_count",
        "flag_author_name_variants_in_paper"
    ]
].copy()

display(author_problem_examples.head(20))


In [ ]:
cols_for_next_steps = [
    "id",
    "title",
    "title_norm",
    "abstract",
    "abstract_norm",
    "keywords",
    "keywords_norm",
    "year",
    "year_clean",
    "authors",
    "author_names_raw",
    "author_names_norm",
    "author_names_ascii",
    "author_ids_clean",
    "author_orgs_clean",
    "n_authors",
    "references",
    "n_references",
    "lang",
    "lang_norm",
    "doi",
    "doi_norm",
    "venue",
    "venue_clean",
    "venue_norm",
    "doc_type",
    "doc_type_clean",
    "doc_type_norm",
    "n_keywords",
    "quality_score"
] + quality_cols

df_model_base = df_clean[cols_for_next_steps].copy()
print(df_model_base.shape)
display(df_model_base.head(3))